# DT4 agent: the forest biomass twin over MCP

DT4 serves the forest above-ground biomass (AGB) and carbon twin as an MCP server, `dt4_server.py`.
It observes woodland with **ESA Biomass CCI v7** (100 m, 2005-2012 and 2015-2024; the only data
it is calibrated against), grows stands with two interchangeable growth models, **Chapman-Richards
yield curves** indexed by yield class and the **3-PG** process model, and applies management and
disturbance scenarios with full carbon bookkeeping (living biomass, deadwood, litter, harvested
wood products). Pilot landscapes: **Thetford** (mainly Corsican pine) and **Kielder** (mainly
Sitka spruce). Woodland, species and stand ages come from the **National Forest Inventory**
woodland map and the **Forestry England sub-compartment database**, and every species group is
calibrated to CCI on its own; climate for 3-PG is **HadUK-Grid** 1991-2020.

This notebook acts as the orchestrator: it launches the server over stdio, lists its tools and
calls each of them, exactly as an orchestrating agent would.

In [1]:
import sys
from pathlib import Path

from mcp import Client, StdioServerParameters

SERVER = Path("dt4_server.py").resolve()
params = StdioServerParameters(command=sys.executable, args=[str(SERVER)])


async def ask(tool, **args):
    """Launch DT4, call one tool and return its structured result."""
    async with Client(params) as client:
        r = await client.call_tool(tool, args)
        if r.is_error:
            raise RuntimeError(r.content[0].text)
        return r.structured_content

## Discover the tools

In [2]:
async with Client(params) as client:
    for t in (await client.list_tools()).tools:
        print(f"{t.name:24s} {t.description[:88]}")

list_species             List the species, management regimes, disturbance types and growth models DT4 knows, wit
list_landscapes          List the pilot landscapes with their woodland area, species composition, CCI v7 years, c
forest_inventory         Forest inventory of a landscape or area: woodland area by species group with stand ages 
get_agb                  Observed above-ground biomass (Mg/ha) of the woodland in a landscape, or part of it, in 
agb_timeseries           Annual CCI v7 AGB of a landscape or area for every year with a map (2005-2012, 2015-2024
carbon_stock             Living biomass carbon (above- and below-ground) of a landscape or area in a CCI year, pe
biomass_change           AGB change between two CCI years for a landscape or area: mean change with propagated un
attribute_change         Attribute the significant AGB loss between two CCI years to pressures, using evidence yo
simulate_stand           Simulate one stand forward with the yield-curve model, 3-PG, or

## The pilot landscapes: woodland, species and climate

In [3]:
ls = await ask("list_landscapes")
for x in ls["landscapes"]:
    print(f"{x['name']}: {x['woodland_area_ha']:,.0f} ha woodland ({x['woodland_source']}), "
          f"{x['annual_rain_mm']:.0f} mm/yr, {x['mean_temperature_c']:.1f} C (HadUK-Grid)")
    for r in x["composition"]:
        print(f"   {r['species']} {r['name']:38s} {r['share']:5.1%}  age {r['age_median'] or 0:4.0f}  "
              f"FE YC {r['fe_yield_class_median'] or 0:4.0f}  calibrated {r['calibrated']}")

Thetford Forest: 13,670 ha woodland (NFI woodland GB 2024), 648 mm/yr, 10.2 C (HadUK-Grid)
   CP Corsican pine                          56.4%  age   37  FE YC   14  calibrated True
   MB Mixed and other broadleaves            15.3%  age   62  FE YC    4  calibrated True
   SP Scots pine                             15.0%  age   46  FE YC   10  calibrated True
   DF Douglas fir and other conifers          4.6%  age   17  FE YC   12  calibrated True
   BE Beech                                   3.6%  age   82  FE YC    6  calibrated True
   LA Larches (hybrid, European, Japanese)    1.9%  age   15  FE YC   10  calibrated True
   BI Birch (downy and silver)                1.5%  age   21  FE YC    4  calibrated True
   OK Oak                                     1.5%  age   84  FE YC    4  calibrated True
   NS Norway spruce (and other spruces)       0.0%  age    8  FE YC   10  calibrated False
   LP Lodgepole pine                          0.0%  age   54  FE YC   10  calibrated False
Kielder

## CCI v7 calibration per species group (FE yield class only as a check)

In [4]:
inv = await ask("forest_inventory", landscape="kielder")
for c in inv["calibration"]:
    h, f = c["holdout_2018_2024"], c["holdout_2018_2024_free_age"]
    fe = f" | FE YC {c['fe_yield_class_median']:.0f}, r {c['fe_yield_class_pearson_r']:+.2f}" if c["fe_yield_class_median"] else ""
    print(f"{c['species']} {c['growth_model']:12s} median {c['productivity_median']:5.2f} "
          f"(IQR {c['productivity_p25']}-{c['productivity_p75']}), age {c['stand_age_median_2024']:3.0f}, "
          f"hold-out RMSE {h['rmse_mg_ha']:.1f} (CCI-fitted ages {f['rmse_mg_ha']:.1f}){fe}")

SS yield_curve  median  8.00 (IQR 2.0-17.0), age  28, hold-out RMSE 32.5 (CCI-fitted ages 33.1) | FE YC 12, r +0.38
SS 3pg          median  0.20 (IQR 0.0-0.5), age  28, hold-out RMSE 31.2 (CCI-fitted ages 33.5)
NS yield_curve  median  4.00 (IQR 2.0-13.0), age  25, hold-out RMSE 30.1 (CCI-fitted ages 34.9) | FE YC 10, r +0.35
NS 3pg          median  0.10 (IQR 0.0-0.5), age  25, hold-out RMSE 29.2 (CCI-fitted ages 35.7)
MB yield_curve  median  2.00 (IQR 2.0-3.0), age  19, hold-out RMSE 23.4 (CCI-fitted ages 18.0) | FE YC 2, r +0.05
MB 3pg          median  0.15 (IQR 0.1-0.3625), age  19, hold-out RMSE 18.8 (CCI-fitted ages 18.0)
BI yield_curve  median  3.00 (IQR 2.0-4.0), age  12, hold-out RMSE 34.9 (CCI-fitted ages 47.5) | FE YC 4, r -0.08
BI 3pg          median  0.40 (IQR 0.25-0.5), age  12, hold-out RMSE 20.8 (CCI-fitted ages 45.0)
LP yield_curve  median  3.00 (IQR 2.0-6.0), age  59, hold-out RMSE 37.5 (CCI-fitted ages 31.4) | FE YC 6, r +0.41
LP 3pg          median  0.10 (IQR 0.05-0.2

## State: observed AGB and living carbon (EBCF role: state)

In [5]:
for key in ("thetford", "kielder"):
    a = await ask("get_agb", landscape=key, year=2024)
    c = await ask("carbon_stock", landscape=key, year=2024)
    print(f"{key:9s} 2024 AGB {a['agb_mean_mg_ha']:.1f} Mg/ha (90% {a['agb_p05_mg_ha']:.1f}-{a['agb_p95_mg_ha']:.1f}), "
          f"living C {c['living_c_t_ha']:.1f} t C/ha, total {c['total_living_co2e_t']/1e6:.2f} Mt CO2e")

ts = await ask("agb_timeseries", landscape="kielder")
print("\nKielder CCI v7 series (Mg/ha):", " ".join(f"{r['year']}:{r['agb_mean_mg_ha']:.0f}" for r in ts["series"]))
print(f"trend 2015-2024: {ts['trend_mg_ha_yr_2015_2024']:+.2f} Mg/ha/yr;", ts["sensor_epoch_note"])

for sp in ("SS", "NS", "LP"):
    a = await ask("get_agb", landscape="kielder", year=2024, species=sp)
    print(f"Kielder {sp} 2024: {a['agb_mean_mg_ha']:.1f} Mg/ha over {a['area_ha']:,.0f} ha")

pt = await ask("get_agb", landscape="kielder", year=2024, area={"point": [-2.55, 55.2], "radius_m": 500})
print(f"\nOne stand: {pt['area']}: {pt['agb_mean_mg_ha']:.1f} +- {pt['agb_sd_mg_ha']:.1f} Mg/ha over {pt['n_pixels']} pixels")

thetford  2024 AGB 102.9 Mg/ha (90% 82.8-123.0), living C 66.2 t C/ha, total 3.32 Mt CO2e


kielder   2024 AGB 81.4 Mg/ha (90% 60.8-101.9), living C 52.3 t C/ha, total 5.09 Mt CO2e



Kielder CCI v7 series (Mg/ha): 2005:64 2006:63 2007:64 2008:63 2009:64 2010:63 2011:64 2012:64 2015:72 2016:74 2017:75 2018:77 2019:78 2020:81 2021:82 2022:83 2023:83 2024:81
trend 2015-2024: +1.31 Mg/ha/yr; 2005-2012 maps use ALOS PALSAR/Envisat, 2015-2024 ALOS-2/Sentinel-1; compare levels across the gap with care.


Kielder SS 2024: 86.8 Mg/ha over 22,336 ha


Kielder NS 2024: 62.6 Mg/ha over 1,520 ha


Kielder LP 2024: 87.3 Mg/ha over 378 ha



One stand: point lon -2.5500, lat 55.2000, radius 500 m: 77.8 +- 29.8 Mg/ha over 86 pixels


## Impact and pressure: what changed, and why (EBCF roles: impact, pressure)

In [6]:
ch = await ask("biomass_change", landscape="kielder", year1=2015, year2=2024)
print(f"Kielder 2015-2024: {ch['delta_agb_mean_mg_ha']:+.1f} +- {ch['delta_agb_sd_mg_ha']:.1f} Mg/ha -> '{ch['impact']}'")
print(f"  significant loss on {ch['loss_area_ha']:,.0f} ha ({ch['conversion_candidate_area_ha']:,.0f} ha to below the woodland threshold), "
      f"gain on {ch['gain_area_ha']:,.0f} ha")

# attribution needs independent evidence; here a made-up felling footprint over the western half
at = await ask("attribute_change", landscape="kielder", year1=2015, year2=2024,
               evidence=[{"pressure": "harvest", "year_from": 2015, "year_to": 2024,
                          "bbox": [-2.70, 55.13, -2.525, 55.30], "source": "illustrative felling layer"}])
for r in at["attributions"]:
    print(f"  {r['label']}: {r['attributed_loss_area_ha']:,.0f} ha ({r['share_of_loss']:.0%})")
print(f"  unattributed: {at['unattributed_share']:.0%}  ({at['rule']})")

Kielder 2015-2024: +9.7 +- 17.6 Mg/ha -> 'no detectable change'
  significant loss on 465 ha (262 ha to below the woodland threshold), gain on 476 ha


  woody vegetation amount decrease (harvest): 357 ha (77%)
  unattributed: 23%  (A pressure is named only where independent evidence overlaps a significant loss in space and time (Living Earth EBCF).)


## Response: one stand, both growth models side by side

In [7]:
s = await ask("simulate_stand", landscape="kielder", age=20, years=50, regime="clearfell_no_thin")
for r in s["runs"]:
    agb = r["agb_mg_ha"]["mean"]
    print(f"{r['growth_model']:12s} {r['productivity_name']} {r['productivity']:.2f}: AGB "
          + " ".join(f"{agb[i]:.0f}" for i in range(0, 51, 10))
          + f" (every 10 yrs); cumulative net {r['cumulative_net_t_co2_ha'][0]:.0f} t CO2/ha "
          f"[{r['cumulative_net_t_co2_ha'][1]:.0f}, {r['cumulative_net_t_co2_ha'][2]:.0f}], "
          f"to soil (DT7) {r['litter_and_deadwood_to_soil_t_c_ha_yr']:.2f} t C/ha/yr")

yield_curve  yield_class 8.00: AGB 56 117 177 2 31 86 (every 10 yrs); cumulative net -366 t CO2/ha [-840, -72], to soil (DT7) 0.40 t C/ha/yr
3pg          fertility_rating 0.20: AGB 66 120 170 4 40 94 (every 10 yrs); cumulative net -307 t CO2/ha [-689, -55], to soil (DT7) 0.24 t C/ha/yr


## Response: management regimes for a typical stand

In [8]:
for key, sp in (("kielder", "SS"), ("kielder", "NS"), ("thetford", "CP"), ("thetford", "SP")):
    m = await ask("management_scenario", landscape=key, species=sp, horizon_years=60)
    print(f"{key} {sp}: start age {m['start_age']:.0f}, start AGB {m['start_agb_mg_ha']:.0f} Mg/ha; best for carbon {m['best_for_carbon']}")
    for o in m["outcomes"]:
        c = o["cumulative_net_t_co2_ha"]
        print(f"   {o['growth_model']:12s} {o['regime']:18s} net {c[0]:7.0f} t CO2/ha [{c[1]:6.0f},{c[2]:6.0f}]  "
              f"logs {o['removals_total_t_dm_ha']:5.0f} t/ha  final C {o['final_total_c_t_ha']:5.0f} t/ha")

kielder SS: start age 28, start AGB 87 Mg/ha; best for carbon {'yield_curve': 'continuous_cover', '3pg': 'no_intervention'}
   yield_curve  no_intervention    net    -651 t CO2/ha [ -1252,  -206]  logs     0 t/ha  final C   201 t/ha
   yield_curve  thin_and_fell      net    -239 t CO2/ha [  -701,    91]  logs   212 t/ha  final C   108 t/ha
   yield_curve  clearfell_no_thin  net    -477 t CO2/ha [ -1219,    26]  logs   141 t/ha  final C   168 t/ha
   yield_curve  continuous_cover   net    -701 t CO2/ha [ -1728,   -35]  logs   172 t/ha  final C   216 t/ha
   3pg          no_intervention    net    -593 t CO2/ha [ -1301,  -146]  logs     0 t/ha  final C   212 t/ha
   3pg          thin_and_fell      net    -268 t CO2/ha [  -805,   109]  logs   248 t/ha  final C   121 t/ha
   3pg          clearfell_no_thin  net    -407 t CO2/ha [ -1050,    52]  logs   156 t/ha  final C   158 t/ha
   3pg          continuous_cover   net    -526 t CO2/ha [ -1307,   -30]  logs   170 t/ha  final C   193 t/ha


kielder NS: start age 25, start AGB 63 Mg/ha; best for carbon {'yield_curve': 'no_intervention', '3pg': 'no_intervention'}
   yield_curve  no_intervention    net    -551 t CO2/ha [ -1183,  -220]  logs     0 t/ha  final C   166 t/ha
   yield_curve  thin_and_fell      net    -155 t CO2/ha [  -476,    36]  logs   144 t/ha  final C    73 t/ha
   yield_curve  clearfell_no_thin  net    -265 t CO2/ha [  -699,   -28]  logs   141 t/ha  final C    97 t/ha
   yield_curve  continuous_cover   net    -495 t CO2/ha [ -1274,   -87]  logs   121 t/ha  final C   154 t/ha
   3pg          no_intervention    net    -488 t CO2/ha [ -1154,  -146]  logs     0 t/ha  final C   169 t/ha
   3pg          thin_and_fell      net    -192 t CO2/ha [  -646,    64]  logs   182 t/ha  final C    86 t/ha
   3pg          clearfell_no_thin  net    -276 t CO2/ha [  -782,     5]  logs   161 t/ha  final C   107 t/ha
   3pg          continuous_cover   net    -419 t CO2/ha [ -1132,   -54]  logs   129 t/ha  final C   149 t/ha


thetford CP: start age 38, start AGB 108 Mg/ha; best for carbon {'yield_curve': 'continuous_cover', '3pg': 'no_intervention'}
   yield_curve  no_intervention    net    -796 t CO2/ha [ -1160,  -253]  logs     0 t/ha  final C   247 t/ha
   yield_curve  thin_and_fell      net    -378 t CO2/ha [  -733,    84]  logs   269 t/ha  final C   156 t/ha
   yield_curve  clearfell_no_thin  net    -665 t CO2/ha [ -1162,    12]  logs   194 t/ha  final C   227 t/ha
   yield_curve  continuous_cover   net   -1175 t CO2/ha [ -1999,   -56]  logs   257 t/ha  final C   343 t/ha
   3pg          no_intervention    net    -976 t CO2/ha [ -1591,  -189]  logs     0 t/ha  final C   330 t/ha
   3pg          thin_and_fell      net    -427 t CO2/ha [  -844,    71]  logs   349 t/ha  final C   176 t/ha
   3pg          clearfell_no_thin  net    -583 t CO2/ha [ -1051,    22]  logs   237 t/ha  final C   218 t/ha
   3pg          continuous_cover   net    -762 t CO2/ha [ -1308,   -53]  logs   230 t/ha  final C   272 t/ha


thetford SP: start age 43, start AGB 104 Mg/ha; best for carbon {'yield_curve': 'continuous_cover', '3pg': 'no_intervention'}
   yield_curve  no_intervention    net    -655 t CO2/ha [ -1052,  -259]  logs     0 t/ha  final C   212 t/ha
   yield_curve  thin_and_fell      net    -172 t CO2/ha [  -478,    97]  logs   188 t/ha  final C   102 t/ha
   yield_curve  clearfell_no_thin  net    -330 t CO2/ha [  -714,    16]  logs   180 t/ha  final C   137 t/ha
   yield_curve  continuous_cover   net    -782 t CO2/ha [ -1599,   -75]  logs   193 t/ha  final C   246 t/ha
   3pg          no_intervention    net    -591 t CO2/ha [ -1067,   -82]  logs     0 t/ha  final C   226 t/ha
   3pg          thin_and_fell      net    -232 t CO2/ha [  -575,   136]  logs   250 t/ha  final C   123 t/ha
   3pg          clearfell_no_thin  net    -323 t CO2/ha [  -685,    82]  logs   201 t/ha  final C   148 t/ha
   3pg          continuous_cover   net    -489 t CO2/ha [  -939,    38]  logs   174 t/ha  final C   197 t/ha


## Response: new woodland (the hook DT10 calls)

In [9]:
for key, sp in (("kielder", "SS"), ("thetford", "OK"), ("thetford", "BI")):
    a = await ask("afforestation_scenario", landscape=key, species=sp, area_ha=100, horizon_years=50)
    print(key, sp, {m: f"{v:.1f} t CO2/ha/yr" for m, v in a["carbon_per_ha_per_year_t_co2"].items()},
          {m: f"{v[0]/1000:.1f} kt CO2 on 100 ha" for m, v in a["total_cumulative_net_t_co2"].items()})

kielder SS {'yield_curve': '-12.9 t CO2/ha/yr', '3pg': '-11.2 t CO2/ha/yr'} {'yield_curve': '-64.4 kt CO2 on 100 ha', '3pg': '-55.8 kt CO2 on 100 ha'}


thetford OK {'yield_curve': '-5.1 t CO2/ha/yr', '3pg': '-5.4 t CO2/ha/yr'} {'yield_curve': '-25.6 kt CO2 on 100 ha', '3pg': '-26.8 kt CO2 on 100 ha'}


thetford BI {'yield_curve': '-12.8 t CO2/ha/yr', '3pg': '-8.0 t CO2/ha/yr'} {'yield_curve': '-63.9 kt CO2 on 100 ha', '3pg': '-40.1 kt CO2 on 100 ha'}


## Which growth model predicts CCI better? Hold-out validation

In [10]:
v = await ask("validate")
for r in v["rows"]:
    h, f = r["holdout"], r["holdout_free_age"]
    print(f"{r['landscape']:9s} {r['species']} {r['growth_model']:12s} RMSE {h['rmse_mg_ha']:5.1f}  "
          f"bias {h['bias_mg_ha']:+6.1f}  (CCI-fitted ages: RMSE {f['rmse_mg_ha']:5.1f})")
print("better:", v["better_model"])

thetford  CP yield_curve  RMSE  37.2  bias   +1.7  (CCI-fitted ages: RMSE  34.2)
thetford  CP 3pg          RMSE  35.7  bias   +2.4  (CCI-fitted ages: RMSE  33.5)
thetford  MB yield_curve  RMSE  30.2  bias   +3.9  (CCI-fitted ages: RMSE  27.9)
thetford  MB 3pg          RMSE  28.8  bias   -5.2  (CCI-fitted ages: RMSE  26.5)
thetford  SP yield_curve  RMSE  45.9  bias  -11.8  (CCI-fitted ages: RMSE  38.9)
thetford  SP 3pg          RMSE  43.3  bias  -11.6  (CCI-fitted ages: RMSE  37.4)
thetford  DF yield_curve  RMSE  41.5  bias  -16.2  (CCI-fitted ages: RMSE  38.5)
thetford  DF 3pg          RMSE  41.2  bias  -16.1  (CCI-fitted ages: RMSE  38.7)
thetford  BE yield_curve  RMSE  47.7  bias   -4.7  (CCI-fitted ages: RMSE  37.6)
thetford  BE 3pg          RMSE  51.4  bias  -29.5  (CCI-fitted ages: RMSE  38.2)
thetford  LA yield_curve  RMSE  63.8  bias  -38.7  (CCI-fitted ages: RMSE  56.0)
thetford  LA 3pg          RMSE  52.6  bias  -30.6  (CCI-fitted ages: RMSE  56.8)
thetford  BI yield_curve  RM

## Learning a site's productivity with the Kalman filter

In [11]:
a = await ask("assimilate_agb", landscape="kielder", site="demo-stand",
              area={"point": [-2.55, 55.2], "radius_m": 300}, persist=False)
for r in a["results"]:
    print(f"{r['growth_model']:12s} prior {r['prior_productivity']:.2f} +- {r['prior_productivity_sd']:.2f} -> "
          f"posterior {r['posterior_productivity']:.2f} +- {r['posterior_productivity_sd']:.2f}  ({r['data']})")

for m in ("yield_curve", "3pg"):
    s = await ask("self_learn_synthetic", growth_model=m)
    print(f"synthetic {m:12s} truth {s['true_productivity']:.2f}, prior {s['prior_productivity']:.2f} -> "
          f"recovered {s['recovered_productivity']:.2f} +- {s['recovered_sd']:.2f}")

yield_curve  prior 8.00 +- 11.12 -> posterior 6.52 +- 2.19  (CCI v7 2005-2024, point lon -2.5500, lat 55.2000, radius 300 m)
3pg          prior 0.20 +- 0.37 -> posterior 0.17 +- 0.08  (CCI v7 2005-2024, point lon -2.5500, lat 55.2000, radius 300 m)


synthetic yield_curve  truth 20.20, prior 9.80 -> recovered 19.06 +- 2.00


synthetic 3pg          truth 0.70, prior 0.30 -> recovered 0.69 +- 0.06
